## Attention Mechanism

### Implementing Self Attention Mechanism

#### Without Training Weights - Simplified Self Attention

In [ ]:
import torch

inputs=torch.tensor([
  [0.43, 0.15, 0.89], # Your     (x^1)
  [0.55, 0.87, 0.66], # journey  (x^2)
  [0.57, 0.85, 0.64], # starts   (x^3)
  [0.22, 0.58, 0.33], # with     (x^4)
  [0.77, 0.25, 0.10], # one      (x^5)
  [0.05, 0.80, 0.55]  # step     (x^6)
])

In [ ]:
import matplotlib.pyplot as plt

words=['Your', 'journey', 'starts', 'with', 'one', 'step']

x_coord=inputs[:, 0].numpy()
y_coord=inputs[:, 1].numpy()
z_coord=inputs[:, 2].numpy()

fig=plt.figure()
ax=fig.add_subplot(111, projection='3d')

for x, y, z, word in zip(x_coord, y_coord, z_coord, words):
  ax.scatter(x, y, z)
  ax.text(x, y, z, word, fontsize=10)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')

plt.title('3D Plot of Word Embeddings')
plt.show()

In [ ]:
fig=plt.figure()
ax=fig.add_subplot(111, projection='3d')

colors=['r', 'g', 'b', 'c', 'm', 'y']

for x, y, z, word, color in zip(x_coord, y_coord, z_coord, words, colors):
  # Draw vector from origin to point (x,y,z) with specified color
  ax.quiver(0, 0, 0, x, y, z, color=color, arrow_length_ratio=0.05)
  ax.text(x, y, z, word, fontsize=10)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')

ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.set_zlim([0, 1])

plt.title('3D Plot of Word Embeddings with Colored Vectors')
plt.show()

In [ ]:
# Calculating the attention scores

# Consider 'journey' as the query
query=inputs[1]
attn_scores=torch.empty(inputs.shape[0])

for i, x_i in enumerate(inputs):
  attn_scores[i]=torch.dot(x_i, query)

print(f"Attention scores w.r.t journey: {attn_scores}")

In [ ]:
# Normalization: To obtain attention weights that sum up to 1
# This is useful for interpretation and maintaining training stability
# Normalized Attention scores: Attention weights

norm_attn_scores=attn_scores/attn_scores.sum()
print(f"Normalized Attention scores: {norm_attn_scores}")
print(norm_attn_scores.sum())

In [ ]:
# Normalization with naive softmax

def softmax_naive(x):
  return torch.exp(x)/torch.exp(x).sum(dim=0) # summing all elements of row

naive_norm_attn_scores=softmax_naive(attn_scores)
print(f"Naive Softmax Normalized Attention Scores: {naive_norm_attn_scores}")

In [ ]:
# Normalization with PyTorch softmax
# This is recommended as reduces numerical instability

pyt_sm_norm_attn_scores=torch.softmax(attn_scores, dim=0)
print(f"PyTorch Softmax Normalized Attention Scores: {pyt_sm_norm_attn_scores}")

In [ ]:
# Context vector of 'journey'

query=inputs[1]
context_vector=torch.zeros(query.shape)

for score, x_i in zip(pyt_sm_norm_attn_scores, inputs):
  context_vector+=score*x_i

print(f"Context vector of journey: {context_vector}")

In [ ]:
inputs_context=torch.tensor([
  [0.43, 0.15, 0.89],  # Your     (x^1)
  [0.55, 0.87, 0.66],  # journey  (x^2)
  [0.57, 0.85, 0.64],  # starts   (x^3)
  [0.22, 0.58, 0.33],  # with     (x^4)
  [0.77, 0.25, 0.10],  # one      (x^5)
  [0.05, 0.80, 0.55],  # step     (x^6)
  [0.4419, 0.6515, 0.5683]
])

words=['Your', 'journey', 'starts', 'with', 'one', 'step', 'journey-context']

x_coord=inputs_context[:, 0].numpy()
y_coord=inputs_context[:, 1].numpy()
z_coord=inputs_context[:, 2].numpy()

fig=plt.figure()
ax=fig.add_subplot(111, projection='3d')

colors=['r', 'g', 'b', 'c', 'm', 'y', 'r']

for x, y, z, word, color in zip(x_coord, y_coord, z_coord, words, colors):
  # Draw vector from origin to point (x,y,z) with specified color
  ax.quiver(0, 0, 0, x, y, z, color=color, arrow_length_ratio=0.05)
  ax.text(x, y, z, word, fontsize=10)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')

ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.set_zlim([0, 1])

plt.title('3D Plot of Word Embeddings with Colored Vectors')
plt.show()

In [ ]:
# Attention Score Matrix

attn_scores_matrix=torch.empty(inputs.shape[0], inputs.shape[0])

for i, x_i in enumerate(inputs):
  for j, x_j in enumerate(inputs):
    attn_scores_matrix[i, j]=torch.dot(x_i, x_j)

print(f"Attention Score Matrix: {attn_scores_matrix}")

In [ ]:
# Computationally efficient - Matrix Multiplication

attn_scores_matrix=torch.matmul(inputs, inputs.T)
print(f"Attention Score Matrix: {attn_scores_matrix}")

- The `dim` parameter in torch.softmax specifies the dimension of the input tensor along which the function will be computed.

- `dim=-1`: softmax will apply the normalization along the last dimension of the attention scores matrix.

In [ ]:
# Attention Weight Matrix

attn_weights_matrix=torch.softmax(attn_scores_matrix, dim=-1)
print(f"Attention Weight Matrix: {attn_weights_matrix}")

In [ ]:
print(f"All Rows Sum: {attn_weights_matrix.sum(dim=-1)}")

In [ ]:
# Context Vector Matrix

context_vector_matrix=torch.matmul(attn_weights_matrix, inputs)
print(f"Context Vector Matrix: {context_vector_matrix}")

#### With Training Weights - Self Attention `(query-key-value)`

In [ ]:
import torch

inputs=torch.tensor([
  [0.43, 0.15, 0.89], # Your     (x^1)
  [0.55, 0.87, 0.66], # journey  (x^2)
  [0.57, 0.85, 0.64], # starts   (x^3)
  [0.22, 0.58, 0.33], # with     (x^4)
  [0.77, 0.25, 0.10], # one      (x^5)
  [0.05, 0.80, 0.55]  # step     (x^6)
])

In [ ]:
d_in=inputs.shape[1]
d_out=2

In [ ]:
torch.manual_seed(123)

w_query=torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
w_key=torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
w_value=torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [ ]:
print(f"Query Trainable Weight Matrix:\n{w_query}")

In [ ]:
print(f"Key Trainable Weight Matrix:\n{w_key}")

In [ ]:
print(f"Value Trainable Weight Matrix:\n{w_value}")

- We are setting `requires_grad=False` to reduce clutter in outputs for illustration purposes.

- If we were using the weight matrices for model training, we will set `requires_grad=True` to update these matrices during training.

In [ ]:
# We will consider 'journey' as the query

x_2=inputs[1]

query_2=torch.matmul(x_2, w_query)
key_2=torch.matmul(x_2, w_key)
value_2=torch.matmul(x_2, w_value)

print(query_2)

In [ ]:
queries=torch.matmul(inputs, w_query)
keys=torch.matmul(inputs, w_key)
values=torch.matmul(inputs, w_value)

print(f"{queries}\n{keys}\n{values}")

In [ ]:
# Attention Score for journey
attn_score_2=torch.matmul(query_2, keys.T)
print(f"Attention Scores for journey: {attn_score_2}")

In [ ]:
# Attention Score Matrix
attn_scores_matrix=torch.matmul(queries, keys.T)
print(f"Attention Score Matrix:\n{attn_scores_matrix}")

- Now we will normalize the attention scores by dividing them by the square root of the `embedding dimension` of the keys.

- Why to divide by `sqrt(dim)`:
  * `For stability in learning`: Softmax function is sensitive to magnitudes of its inputs. When the inputs are large, the differences between the exponential values of each input become much more pronounced. This causes softmax output to become `peaky`, where the highest value receives most probability mass and the rest receives very little. In transformers, if the dot products of query and key vectors become too large, the attention scores can become `very large`. This results in a `very sharp softmax distribution`, making the model `overly confident` in one particular key. Such sharp distributions can make learning unstable.

  * `To make the variance of dot product stable`: The dot product of Q and K `increases` the variance because multiplying 2 random numbers increases the variance. The increase in variance `grows` with the dimension. Dividing by sqrt(dim) keeps the variance `close to 1`.

In [ ]:
# Attention Weight for journey
d_k=keys.shape[-1]

attn_weights_2=torch.softmax(attn_score_2/d_k**0.5, dim=-1)
print(f"Attention weights for journey: {attn_weights_2}")

In [ ]:
# Attention Weight Matrix
attn_weights_matrix=torch.softmax(attn_scores_matrix/d_k**0.5, dim=-1)
print(f"Attention Weight Matrix:\n{attn_weights_matrix}")

In [ ]:
# Context Vector for journey
context_vector_2=torch.matmul(attn_weights_2, values)
print(f"Context Vector for journey: {context_vector_2}")

In [ ]:
# Context Vector Matrix
context_vector_matrix=torch.matmul(attn_weights_matrix, values)
print(f"Context Vector Matrix:\n{context_vector_matrix}")

#### Self Attention Class - Basic

In [ ]:
import torch.nn as nn

class SelfAttentionBasic(nn.Module):
  def __init__(self, d_in, d_out):
    super().__init__()
    self.w_query=nn.Parameter(torch.rand(d_in, d_out))
    self.w_key=nn.Parameter(torch.rand(d_in, d_out))
    self.w_value=nn.Parameter(torch.rand(d_in, d_out))

  def forward(self, x):
    queries=x@self.w_query
    keys=x@self.w_key
    values=x@self.w_value

    attn_scores=queries@keys.T
    attn_weights=torch.softmax(attn_scores/keys.shape[-1]**0.5, dim=-1)
    context_vectors=attn_weights@values

    return context_vectors

- `nn.Module` from PyTorch is a fundamental building block of PyTorch Models, which provides necessary functionalities for model layer creation and management.

In [ ]:
torch.manual_seed(123)

sa_basic=SelfAttentionBasic(d_in, d_out)
print(sa_basic(inputs))

#### Self Attention Class - Advanced

In [ ]:
import torch.nn as nn

class SelfAttentionAdvanced(nn.Module):
  def __init__(self, d_in, d_out, qkv_bias=False):
    super().__init__()
    self.w_query=nn.Linear(d_in, d_out, bias=qkv_bias)
    self.w_key=nn.Linear(d_in, d_out, bias=qkv_bias)
    self.w_value=nn.Linear(d_in, d_out, bias=qkv_bias)

  def forward(self, x):
    queries=self.w_query(x)
    keys=self.w_key(x)
    values=self.w_value(x)

    attn_scores=queries@keys.T
    attn_weights=torch.softmax(attn_scores/keys.shape[-1]**0.5, dim=-1)
    context_vectors=attn_weights@values

    return context_vectors

In [ ]:
torch.manual_seed(123)

sa_advanced=SelfAttentionAdvanced(d_in, d_out)
print(sa_advanced(inputs))

- `nn.Linear` will effectively perform matrix multiplication when the bias units are disabled.

- Advantage of using nn.Linear is it has an optimized weight initialization scheme, contributing to more stable and effective model training.

### Implementing Causal Attention Mechanism

In [ ]:
import torch

inputs=torch.tensor([
  [0.43, 0.15, 0.89], # Your     (x^1)
  [0.55, 0.87, 0.66], # journey  (x^2)
  [0.57, 0.85, 0.64], # starts   (x^3)
  [0.22, 0.58, 0.33], # with     (x^4)
  [0.77, 0.25, 0.10], # one      (x^5)
  [0.05, 0.80, 0.55]  # step     (x^6)
])

In [ ]:
import torch.nn as nn

class SelfAttentionAdvanced(nn.Module):
  def __init__(self, d_in, d_out, qkv_bias=False):
    super().__init__()
    self.w_query=nn.Linear(d_in, d_out, bias=qkv_bias)
    self.w_key=nn.Linear(d_in, d_out, bias=qkv_bias)
    self.w_value=nn.Linear(d_in, d_out, bias=qkv_bias)

  def forward(self, x):
    queries=self.w_query(x)
    keys=self.w_key(x)
    values=self.w_value(x)

    attn_scores=queries@keys.T
    attn_weights=torch.softmax(attn_scores/keys.shape[-1]**0.5, dim=-1)
    context_vectors=attn_weights@values

    return context_vectors

In [ ]:
d_in=inputs.shape[1]
d_out=2

In [ ]:
torch.manual_seed(123)
sa_advanced=SelfAttentionAdvanced(d_in, d_out)

queries=sa_advanced.w_query(inputs)
keys=sa_advanced.w_key(inputs)

attn_scores=torch.matmul(queries, keys.T)
attn_weights=torch.softmax(attn_scores/keys.shape[-1]**0.5, dim=1)
print(attn_weights)

In [ ]:
context_length=attn_scores.shape[0]
mask_simple=torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

In [ ]:
masked_attn_weights=attn_weights*mask_simple
print(masked_attn_weights)

In [ ]:
rows_sum=masked_attn_weights.sum(dim=1, keepdim=True)
norm_masked_attn_weights=masked_attn_weights/rows_sum
print(norm_masked_attn_weights)

#### Masking with Upper Triangular Infinity Matrix

- `masked_fill` looks into the matrix where the values are positive and replace them with the value we provided as second argument.

In [ ]:
mask=torch.triu(torch.ones(context_length, context_length), diagonal=1)
inf_mask=attn_scores.masked_fill(mask.bool(), -torch.inf)
print(inf_mask)

In [ ]:
attn_weights=torch.softmax(inf_mask/keys.shape[-1]**0.5, dim=1)
print(attn_weights)

- Masking in transformers sets scores for future tokens to a large negative value, making their influence in the softmax calculation effectively 0.

- The softmax function then recalculates attention weights only among the unmasked tokens.

- This process ensures no information leakage from masked tokens, focusing model solely on the intended data.

#### Masking Additional Attention Weights with Dropout

In [ ]:
torch.manual_seed(123)

dropout=nn.Dropout(0.5)
example=torch.ones(6, 6)
print(dropout(example))

- When applying dropout to an attention weight matrix with a rate of 50%, half of the elements in the matrix are randomly set to 0.

- To compensate for the reduction in active elements, the values of the remaining elements in the matrix are scaled up by a factor of 1/0.5=2.

- This scaling is crucial to maintain the overall balance of the attention weights, ensuring that the average influence of the attention mechanism remains consistent during both training and inference phases.

In [ ]:
torch.manual_seed(123)

dropout_attn_weights=dropout(attn_weights)
print(dropout_attn_weights)

#### Creating batch of inputs

This is a 3D tensor consisting of 2 input texts with 6 tokens each and each token is a 3-dimensional embedding vector.

In [ ]:
batch=torch.stack((inputs, inputs), dim=0)
print(batch.shape)

#### Causal Attention Class

In [ ]:
import torch.nn as nn

class CausalAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
    super().__init__()
    self.d_out=d_out
    self.w_query=nn.Linear(d_in, d_out, bias=qkv_bias)
    self.w_key=nn.Linear(d_in, d_out, bias=qkv_bias)
    self.w_value=nn.Linear(d_in, d_out, bias=qkv_bias)
    self.dropout=nn.Dropout(dropout)
    self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

  def forward(self, x):
    b, num_tokens, d_in=x.shape
    queries=self.w_query(x)
    keys=self.w_key(x)
    values=self.w_value(x)

    attn_scores=torch.matmul(queries, keys.transpose(1, 2))
    # :num_tokens to account for cases where the number of tokens in the batch is smaller than context length
    attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens],
                             -torch.inf)
    
    attn_weights=torch.softmax(attn_scores/keys.shape[-1]**0.5, dim=-1)
    attn_weights=self.dropout(attn_weights)

    context_vectors=torch.matmul(attn_weights, values)
    return context_vectors

In [ ]:
torch.manual_seed(123)
context_length=batch.shape[1]

ca=CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs=ca(batch)
print(context_vecs.shape)

In [ ]:
print(context_vecs)

- The `register_buffer` in PyTorch is not strictly necessary for all use cases but offers several advantages.

- When we use the CausalAttention class in our LLM, buffers are automatically moved to appropriate device (CPU or GPU) along with our model which will be relevant when training the LLM in future.

- This means we dont need to manually ensure these tensors are on the same device as your model parameters, avoiding device mismatch errors.

### Implementing Multi-Head Attention Mechanism

#### Stacking Multiple Single Head Attention Layers

In [ ]:
import torch

In [ ]:
import torch.nn as nn

class CausalAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
    super().__init__()
    self.d_out=d_out
    self.w_query=nn.Linear(d_in, d_out, bias=qkv_bias)
    self.w_key=nn.Linear(d_in, d_out, bias=qkv_bias)
    self.w_value=nn.Linear(d_in, d_out, bias=qkv_bias)
    self.dropout=nn.Dropout(dropout)
    self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

  def forward(self, x):
    b, num_tokens, d_in=x.shape
    queries=self.w_query(x)
    keys=self.w_key(x)
    values=self.w_value(x)

    attn_scores=torch.matmul(queries, keys.transpose(1, 2))
    # :num_tokens to account for cases where the number of tokens in the batch is smaller than context length
    attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens],
                             -torch.inf)
    
    attn_weights=torch.softmax(attn_scores/keys.shape[-1]**0.5, dim=-1)
    attn_weights=self.dropout(attn_weights)

    context_vectors=torch.matmul(attn_weights, values)
    return context_vectors

In [ ]:
import torch.nn as nn

class MultiHeadAttentionWrapper(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
    super().__init__()
    self.heads=nn.ModuleList(
      [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
       for _ in range(num_heads)]
    )

  def forward(self, x):
    final_context_vec=torch.cat([head(x) for head in self.heads], dim=-1)
    return final_context_vec

In [ ]:
batch=torch.stack((inputs, inputs), dim=0)

In [ ]:
torch.manual_seed(123)

context_length=batch.shape[1]
d_in, d_out=inputs.shape[1], 2
num_heads=2

mha=MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads)
context_vecs=mha(batch)

print(context_vecs.shape)
print(context_vecs)

#### Multi-Head Attention with Weight Splits

Instead of maintaining `MultiHeadAttentionWrapper` and `CausalAttention` as seperate classes, we will merge both the functionalities into a single class named `MultiHeadAttention`.

In [ ]:
import torch.nn as nn

class MultiHeadAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
    super().__init__()
    assert (d_out%num_heads==0), \
    "d_out must be divisible by num_heads"

    self.d_out=d_out
    self.num_heads=num_heads
    self.head_dim=d_out//num_heads

    self.w_query=nn.Linear(d_in, d_out, bias=qkv_bias)
    self.w_key=nn.Linear(d_in, d_out, bias=qkv_bias)
    self.w_value=nn.Linear(d_in, d_out, bias=qkv_bias)
    # Linear layer to combine head outputs
    self.out_proj=nn.Linear(d_out, d_out)
    self.dropout=nn.Dropout(dropout)
    self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

  def forward(self, x):
    b, num_tokens, d_in=x.shape

    queries=self.w_query(x)
    keys=self.w_key(x)
    values=self.w_value(x)

    # We implicitly split the matrix by adding a `num_heads` dimension
    # Unroll last dimension: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
    keys=keys.view(b, num_tokens, self.num_heads, self.head_dim)
    queries=queries.view(b, num_tokens, self.num_heads, self.head_dim)
    values=values.view(b, num_tokens, self.num_heads, self.head_dim)

    # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
    keys=keys.transpose(1, 2)
    queries=queries.transpose(1, 2)
    values=values.transpose(1, 2)

    # Computing attention scores
    attn_scores=torch.matmul(queries, keys.transpose(2, 3))

    # Original mask truncated to the number of tokens and converted to bool
    masked_bool=self.mask.bool()[:num_tokens, :num_tokens]

    # Using the mask to fill the attention scores
    masked_attn_scores=attn_scores.masked_fill_(masked_bool, -torch.inf)

    # Calculating the attention weights
    attn_weights=torch.softmax(masked_attn_scores/keys.shape[-1]**0.5, dim=-1)

    # Feeding attention weights to the dropout layer
    attn_weights=self.dropout(attn_weights)

    # Calculating context vectors
    context_vecs=(attn_weights@values).transpose(1, 2) # To get the original dimensions

    # Combining heads where self.d_out=num_heads*head_dim
    # contiguous - to make sure the reshaped matrices are in same blocks of memory
    context_vecs=context_vecs.contiguous().view(b, num_tokens, self.d_out)
    context_vecs=self.out_proj(context_vecs)

    return context_vecs

In [ ]:
torch.manual_seed(123)

inputs=torch.tensor(
  [[0.43, 0.15, 0.89, 0.55, 0.87, 0.66],
   [0.57, 0.85, 0.64, 0.22, 0.58, 0.33],
   [0.77, 0.25, 0.10, 0.05, 0.80, 0.55]]
)

batch=torch.stack((inputs, inputs), dim=0)
print(f"Batch Shape: {batch.shape}")

batch_size, context_length, d_in=batch.shape
d_out=6

mha=MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs=mha(batch)
print(context_vecs)
print(f"Context Vectors Shape: {context_vecs.shape}")

- The smallest GPT-2 model (117 million parameters) has 12 attention heads and a context vector embedding size of 768.

- The largest GPT-2 model (1.5 billion parameters) has 25 attention heads amd a context vector embedding size of 1600.

- The embedding sizes of the input tokens and context embeddings are same in GPT models `(d_in=d_out)`.